In [ ]:
# ============================================================
# BAYAN X
# Evidence-Driven AI Data Incident Investigator
# ============================================================
#
# BAYAN X investigates data-quality incidents by combining:
# - secure input handling,
# - structured incident extraction,
# - model routing,
# - deterministic risk scoring,
# - evidence-based root-cause hypotheses,
# - controlled tool execution,
# - caching,
# - and audit logging.
#
# The goal is to support reliable, explainable, and auditable
# AI-assisted data incident investigation.
# ============================================================


# ------------------------------------------------------------
# 1. IMPORTS
# ------------------------------------------------------------

from dataclasses import dataclass, field
from typing import Protocol
from datetime import datetime

from pydantic import BaseModel, field_validator

import hashlib
import re
import time


# ------------------------------------------------------------
# 2. PROJECT CONFIGURATION
# ------------------------------------------------------------

# Public name of the AI assistant.
ASSISTANT_NAME = "BAYAN X"

# Demo user used by the authorization layer.
CURRENT_USER = "Abeer"

# Supported business/data domains.
# Restricting domains keeps structured outputs predictable.
ALLOWED_DOMAINS = [
    "customers",
    "employees",
    "finance",
    "government_services",
    "operations",
    "other"
]

# Supported categories of data-quality incidents.
ISSUE_TYPES = [
    "missing_values",
    "duplicates",
    "inconsistent_values",
    "invalid_format",
    "data_freshness",
    "other"
]

# Simple prompt-injection indicators used by the input guard.
BLOCKED_PHRASES = [
    "تجاهل التعليمات",
    "ignore previous instructions",
    "show system prompt",
    "system prompt"
]

# Regular expressions used to identify and mask common PII.
PII_PATTERNS = {
    "NATIONAL_ID": r"\b[12]\d{9}\b",
    "PHONE": r"\b05\d{8}\b"
}

# Demo-only secret used to test output-leakage protection.
# Never store real secrets directly in production source code.
SECRET_CODE = "BAYAN-INTERNAL-2026"

print("System:", ASSISTANT_NAME)

System: BAYAN X


In [ ]:
# ------------------------------------------------------------
# 3. STANDARD LLM REQUEST / RESPONSE OBJECTS
# ------------------------------------------------------------

@dataclass
class LLMRequest:
    """
    Standard internal request format.

    Keeping one request format allows BAYAN X to work with
    different model providers without changing the rest
    of the application.
    """

    messages: list[dict]
    model: str
    max_tokens: int = 512


@dataclass
class LLMResponse:
    """
    Standard response returned by every model adapter.

    Extra metadata is captured so model performance and cost
    can be compared later.
    """

    text: str
    provider: str
    latency: float
    estimated_cost: float

In [ ]:
# ------------------------------------------------------------
# 4. BOUNDED CONVERSATION MEMORY
# ------------------------------------------------------------

@dataclass
class ConversationState:
    """
    Stores recent conversation context.

    The history is intentionally bounded so the prompt does
    not grow forever and consume unnecessary context tokens.
    """

    max_turns: int = 8
    messages: list[dict] = field(default_factory=list)

    def add(self, role, content):
        """Add one message to the conversation history."""

        self.messages.append({
            "role": role,
            "content": content
        })

        # One turn usually contains:
        # 1 user message + 1 assistant message.
        # Therefore, 8 turns = maximum 16 messages.
        self.messages = self.messages[
            -(self.max_turns * 2):
        ]

    def get_messages(self):
        """Return the currently retained conversation history."""

        return self.messages

In [ ]:
# ------------------------------------------------------------
# 5. PROVIDER-INDEPENDENT LLM INTERFACE
# ------------------------------------------------------------

class LLMClient(Protocol):
    """
    Defines the contract that every LLM adapter must follow.

    This avoids coupling BAYAN X to one specific provider.
    """

    def complete(
        self,
        request: LLMRequest
    ) -> LLMResponse:
        ...


# ------------------------------------------------------------
# 6. DEMO MODEL ADAPTER
# ------------------------------------------------------------

class DemoLLMClient:
    """
    Simulates an external or local language model.

    No real API call is made in this prototype.
    Latency and cost are simulated for architecture testing.
    """

    def __init__(
        self,
        name,
        latency=0.4,
        price_per_token=2
    ):
        self.name = name
        self.latency = latency
        self.price_per_token = price_per_token

    def complete(
        self,
        request: LLMRequest
    ) -> LLMResponse:

        start = time.time()

        # Use the most recent user message as the demo input.
        question = request.messages[-1]["content"]

        # Simulate network/model processing latency.
        time.sleep(self.latency)

        text = (
            f"{self.name} analyzed the incident: "
            f"{question}"
        )

        # Simplified token estimation for demonstration only.
        tokens = len(
            (question + text).split()
        )

        estimated_cost = (
            tokens * self.price_per_token
        )

        return LLMResponse(
            text=text,
            provider=self.name,
            latency=round(
                time.time() - start,
                2
            ),
            estimated_cost=estimated_cost
        )

In [ ]:
# ------------------------------------------------------------
# 7. MODEL ROUTING
# ------------------------------------------------------------

# Multiple model routes allow BAYAN X to balance
# intelligence, privacy, latency, and cost.
MODEL_ROUTES = {
    "cheap": DemoLLMClient(
        name="CostEfficientModel",
        latency=0.2,
        price_per_token=2
    ),

    "flagship": DemoLLMClient(
        name="FlagshipModel",
        latency=0.5,
        price_per_token=10
    ),

    "local": DemoLLMClient(
        name="Local-vLLM",
        latency=0.3,
        price_per_token=1
    )
}


# Keywords representing potentially complex or sensitive cases.
COMPLEX_WORDS = [
    "critical",
    "privacy",
    "حوكمة",
    "تسريب",
    "حساسة",
    "production"
]


def choose_model(message):
    """
    Route sensitive or complex incidents to the stronger model.

    Routine incidents use the lower-cost route.
    """

    message_lower = message.lower()

    for word in COMPLEX_WORDS:
        if word.lower() in message_lower:
            return "flagship"

    return "cheap"

In [ ]:
# ------------------------------------------------------------
# 8. INPUT SECURITY GUARD
# ------------------------------------------------------------

def check_input(message):
    """
    Reject obvious prompt-injection attempts before the
    message reaches the language model.
    """

    message_lower = message.lower()

    for phrase in BLOCKED_PHRASES:
        if phrase.lower() in message_lower:
            return False, "prompt_injection"

    return True, "ok"


# ------------------------------------------------------------
# 9. PII MASKING
# ------------------------------------------------------------

def mask_pii(message):
    """
    Replace identifiable personal information with labels.

    The goal is to reduce exposure of sensitive information
    before user text reaches an external model.
    """

    masked = message

    for name, pattern in PII_PATTERNS.items():

        masked = re.sub(
            pattern,
            f"[{name}]",
            masked
        )

    return masked


# ------------------------------------------------------------
# 10. OUTPUT SECURITY GUARD
# ------------------------------------------------------------

def check_output(answer):
    """
    Prevent protected internal information from being returned
    to the user.
    """

    if SECRET_CODE in answer:
        return False, "secret_leak"

    return True, "ok"

In [ ]:
# ------------------------------------------------------------
# 11. STRUCTURED INCIDENT SCHEMA
# ------------------------------------------------------------

class DataIncident(BaseModel):
    """
    Structured representation of an unstructured data issue.

    Converting natural language into validated fields makes the
    incident easier to search, score, route, and automate.
    """

    incident_id: str
    issue_type: str
    data_domain: str
    affected_column: str | None = None
    affected_percentage: int | None = None
    description: str

    @field_validator("issue_type")
    @classmethod
    def validate_issue_type(cls, value):
        """Accept only supported data-quality issue types."""

        if value not in ISSUE_TYPES:
            raise ValueError(
                "Unsupported issue type"
            )

        return value

    @field_validator("data_domain")
    @classmethod
    def validate_domain(cls, value):
        """Accept only known organizational data domains."""

        if value not in ALLOWED_DOMAINS:
            raise ValueError(
                "Unsupported data domain"
            )

        return value

    @field_validator("affected_percentage")
    @classmethod
    def validate_percentage(cls, value):
        """Ensure percentage values remain logically valid."""

        if value is None:
            return value

        if not 0 <= value <= 100:
            raise ValueError(
                "Percentage must be between 0 and 100"
            )

        return value

In [ ]:
# ------------------------------------------------------------
# 12. INCIDENT IDENTIFIER
# ------------------------------------------------------------

def generate_incident_id(message):
    """
    Generate a deterministic identifier from the incident text.

    The same input produces the same demo ID, which is useful
    for reproducible testing.
    """

    digest = hashlib.sha256(
        message.encode("utf-8")
    ).hexdigest()[:6].upper()

    return f"DQ-{digest}"


# ------------------------------------------------------------
# 13. STRUCTURED INCIDENT EXTRACTION
# ------------------------------------------------------------

def extract_incident(message):
    """
    Convert free-form incident text into a validated
    DataIncident object.

    This prototype uses deterministic rules so the extraction
    logic remains transparent during demonstration.
    """

    text = message.lower()

    # ---- Detect the issue category -------------------------

    if (
        "مفقود" in text
        or "missing" in text
        or "null" in text
    ):
        issue_type = "missing_values"

    elif (
        "تكرار" in text
        or "duplicate" in text
        or "مكرر" in text
    ):
        issue_type = "duplicates"

    elif (
        "غير متطابق" in text
        or "inconsistent" in text
    ):
        issue_type = "inconsistent_values"

    else:
        issue_type = "other"

    # ---- Detect the affected business domain --------------

    if "عميل" in text or "customer" in text:
        domain = "customers"

    elif "موظف" in text or "employee" in text:
        domain = "employees"

    elif "مالي" in text or "finance" in text:
        domain = "finance"

    elif "حكومي" in text:
        domain = "government_services"

    else:
        domain = "operations"

    # ---- Extract percentage such as 38% -------------------

    percentage_match = re.search(
        r"(\d{1,3})\s*%",
        message
    )

    percentage = None

    if percentage_match:
        percentage = int(
            percentage_match.group(1)
        )

    # ---- Detect commonly affected columns -----------------

    known_columns = [
        "customer_id",
        "national_id",
        "department",
        "email",
        "phone"
    ]

    affected_column = None

    for column in known_columns:

        if column.lower() in text:
            affected_column = column
            break

    # Pydantic validates the final structured incident.
    return DataIncident(
        incident_id=generate_incident_id(
            message
        ),
        issue_type=issue_type,
        data_domain=domain,
        affected_column=affected_column,
        affected_percentage=percentage,
        description=message
    )

In [ ]:
# ------------------------------------------------------------
# 14. DETERMINISTIC RISK ENGINE
# ------------------------------------------------------------

def calculate_risk(incident):
    """
    Calculate incident risk using explicit business rules.

    The LLM does NOT generate the score.
    This keeps risk assessment explainable and reproducible.
    """

    score = 0
    reasons = []

    # Different issue types contribute different levels of risk.
    if incident.issue_type == "missing_values":

        score += 20
        reasons.append(
            "Missing values detected"
        )

    elif incident.issue_type == "duplicates":

        score += 25
        reasons.append(
            "Duplicate records detected"
        )

    elif incident.issue_type == "inconsistent_values":

        score += 30
        reasons.append(
            "Data inconsistency detected"
        )

    # Large-scale incidents receive additional risk.
    if incident.affected_percentage is not None:

        if incident.affected_percentage >= 30:

            score += 25
            reasons.append(
                "More than 30% of records are affected"
            )

        elif incident.affected_percentage >= 10:

            score += 15
            reasons.append(
                "More than 10% of records are affected"
            )

    # Identity-related fields require stronger protection.
    if incident.affected_column in [
        "national_id",
        "customer_id"
    ]:

        score += 25
        reasons.append(
            "Sensitive identity field affected"
        )

    # Government-service incidents can have wider impact.
    if incident.data_domain == "government_services":

        score += 20
        reasons.append(
            "Government service data affected"
        )

    # Prevent scores from exceeding the defined 100-point scale.
    score = min(score, 100)

    # Convert numerical risk into an operational severity.
    if score >= 80:
        level = "CRITICAL"

    elif score >= 60:
        level = "HIGH"

    elif score >= 30:
        level = "MEDIUM"

    else:
        level = "LOW"

    return {
        "score": score,
        "level": level,
        "reasons": reasons
    }

In [ ]:
# ------------------------------------------------------------
# 15. EVIDENCE-BASED ROOT-CAUSE HYPOTHESES
# ------------------------------------------------------------

def generate_hypotheses(
    incident,
    message
):
    """
    Generate multiple candidate causes instead of pretending
    that the system knows the root cause with certainty.

    Each hypothesis includes:
    - confidence,
    - supporting evidence.
    """

    hypotheses = []
    text = message.lower()

    if incident.issue_type == "missing_values":

        # A recent deployment increases the likelihood that
        # mappings or schemas were changed incorrectly.
        if (
            "تحديث" in text
            or "deployment" in text
            or "update" in text
        ):

            hypotheses.append({
                "hypothesis":
                    "Source-system mapping failure",

                "confidence": 82,

                "evidence": [
                    "Missing values appeared after a system change",
                    "A structured identifier field is affected"
                ]
            })

            hypotheses.append({
                "hypothesis":
                    "Schema mismatch after deployment",

                "confidence": 71,

                "evidence": [
                    "A deployment preceded the anomaly"
                ]
            })

        # Keep an alternative hypothesis when evidence is limited.
        hypotheses.append({
            "hypothesis":
                "Upstream ingestion failure",

            "confidence": 55,

            "evidence": [
                "A large increase in missing values was detected"
            ]
        })

    elif incident.issue_type == "duplicates":

        hypotheses.append({
            "hypothesis":
                "Duplicate ingestion from multiple sources",

            "confidence": 78,

            "evidence": [
                "Duplicate records were detected"
            ]
        })

        hypotheses.append({
            "hypothesis":
                "Missing uniqueness constraint",

            "confidence": 67,

            "evidence": [
                "Repeated identifiers may indicate "
                "missing uniqueness validation"
            ]
        })

    else:

        # Never create false certainty when evidence is weak.
        hypotheses.append({
            "hypothesis":
                "Insufficient evidence",

            "confidence": 30,

            "evidence": [
                "Additional dataset metadata is required"
            ]
        })

    # Present the strongest hypothesis first.
    return sorted(
        hypotheses,
        key=lambda item: item["confidence"],
        reverse=True
    )

In [ ]:
# ------------------------------------------------------------
# 16. CONTROLLED TOOL REGISTRY
# ------------------------------------------------------------

INCIDENT_STATUS = {}


def check_incident_status(reference):
    """Read the current status without modifying system state."""

    return INCIDENT_STATUS.get(
        reference,
        "new"
    )


def assign_incident(
    name,
    reference,
    team
):
    """Assign an incident to an operational team."""

    INCIDENT_STATUS[reference] = (
        f"assigned to {team}"
    )

    return (
        f"{reference} assigned "
        f"to {team} by {name}"
    )


def escalate_incident(
    reference,
    reason
):
    """Escalate a critical incident for human review."""

    INCIDENT_STATUS[reference] = "escalated"

    return (
        f"{reference} escalated: "
        f"{reason}"
    )


# Each tool is registered with an explicit risk classification.
TOOLS = {
    "check_incident_status": {
        "function": check_incident_status,
        "risk": "read_only"
    },

    "assign_incident": {
        "function": assign_incident,
        "risk": "side_effecting"
    },

    "escalate_incident": {
        "function": escalate_incident,
        "risk": "terminal"
    }
}

In [ ]:
# ------------------------------------------------------------
# 17. TOOL AUTHORIZATION GATE
# ------------------------------------------------------------

def run_tool(
    name,
    args
):
    """
    Execute tools through one controlled security gateway.

    The AI should never call operational functions directly.
    """

    # Reject tools that were never explicitly registered.
    if name not in TOOLS:

        return {
            "status": "error",
            "message": "Unknown tool"
        }

    tool = TOOLS[name]

    # Any operation that changes state requires authorization.
    if tool["risk"] == "side_effecting":

        if args.get("name") != CURRENT_USER:

            return {
                "status": "blocked",
                "message": "Not authorized"
            }

    try:

        result = tool["function"](
            **args
        )

        return {
            "status": "success",
            "message": result
        }

    except TypeError:

        # Invalid arguments are handled safely rather than
        # crashing the entire investigation pipeline.
        return {
            "status": "error",
            "message": "Wrong tool arguments"
        }

In [ ]:
# ------------------------------------------------------------
# 18. AUDIT TRAIL
# ------------------------------------------------------------

AUDIT_LOG = []


def log_event(
    incident_id,
    event,
    details
):
    """
    Record significant system decisions.

    Auditability is important because users should be able to
    understand what the AI system did during an investigation.
    """

    AUDIT_LOG.append({
        "timestamp":
            datetime.now().isoformat(
                timespec="seconds"
            ),

        "incident_id": incident_id,
        "event": event,
        "details": details
    })


# ------------------------------------------------------------
# 19. RESPONSE CACHE
# ------------------------------------------------------------

CACHE = {}


def get_cache_key(message):
    """
    Create a stable key for repeated requests.

    Caching avoids unnecessary model calls and therefore
    reduces both latency and model cost.
    """

    return hashlib.sha256(
        message.encode("utf-8")
    ).hexdigest()

In [ ]:
# ------------------------------------------------------------
# 20. MAIN BAYAN X INVESTIGATION PIPELINE
# ------------------------------------------------------------

def investigate(message):
    """
    Execute the complete data-incident investigation workflow.

    Pipeline:
        Security
        → Privacy
        → Cache
        → Structured Extraction
        → Model Routing
        → Risk Assessment
        → Root-Cause Hypotheses
        → Controlled Action
        → Output Safety
        → Audit
    """

    # ========================================================
    # STEP 1 — INPUT SECURITY
    # ========================================================

    allowed, reason = check_input(
        message
    )

    if not allowed:

        return {
            "status": "BLOCKED",
            "reason": reason
        }


    # ========================================================
    # STEP 2 — PRIVACY PROTECTION
    # ========================================================

    # Personal identifiers are removed before the text
    # can reach the model.
    safe_message = mask_pii(
        message
    )


    # ========================================================
    # STEP 3 — CACHE LOOKUP
    # ========================================================

    cache_key = get_cache_key(
        safe_message
    )

    # Reuse an existing safe result for identical requests.
    if cache_key in CACHE:
        return CACHE[cache_key]


    # ========================================================
    # STEP 4 — STRUCTURED EXTRACTION
    # ========================================================

    incident = extract_incident(
        safe_message
    )

    log_event(
        incident.incident_id,
        "incident_created",
        incident.model_dump()
    )


    # ========================================================
    # STEP 5 — MODEL ROUTING
    # ========================================================

    # BAYAN X selects the appropriate model instead of sending
    # every request to the most expensive model.
    selected_route = choose_model(
        safe_message
    )

    client = MODEL_ROUTES[
        selected_route
    ]

    request = LLMRequest(
        messages=[
            {
                "role": "user",
                "content": safe_message
            }
        ],
        model=selected_route
    )

    model_response = client.complete(
        request
    )

    log_event(
        incident.incident_id,
        "model_selected",
        selected_route
    )


    # ========================================================
    # STEP 6 — DETERMINISTIC RISK ASSESSMENT
    # ========================================================

    risk = calculate_risk(
        incident
    )

    log_event(
        incident.incident_id,
        "risk_calculated",
        risk
    )


    # ========================================================
    # STEP 7 — ROOT-CAUSE INVESTIGATION
    # ========================================================

    hypotheses = generate_hypotheses(
        incident,
        safe_message
    )


    # ========================================================
    # STEP 8 — CONTROLLED OPERATIONAL ACTION
    # ========================================================

    action = None

    # Critical incidents are escalated immediately.
    if risk["score"] >= 80:

        action = run_tool(
            "escalate_incident",
            {
                "reference":
                    incident.incident_id,

                "reason":
                    "Critical data risk detected"
            }
        )

    # High-risk incidents are assigned to the specialist team.
    elif risk["score"] >= 60:

        action = run_tool(
            "assign_incident",
            {
                "name":
                    CURRENT_USER,

                "reference":
                    incident.incident_id,

                "team":
                    "Data Quality Team"
            }
        )


    # ========================================================
    # STEP 9 — BUILD EXPLAINABLE RESULT
    # ========================================================

    result = {
        "status":
            "ANALYZED",

        "incident":
            incident.model_dump(),

        "risk":
            risk,

        "model_route":
            selected_route,

        "model_provider":
            model_response.provider,

        "latency_seconds":
            model_response.latency,

        "estimated_cost":
            model_response.estimated_cost,

        "hypotheses":
            hypotheses,

        "action":
            action
    }


    # ========================================================
    # STEP 10 — OUTPUT SECURITY
    # ========================================================

    allowed, reason = check_output(
        str(result)
    )

    if not allowed:

        return {
            "status": "BLOCKED",
            "reason": reason
        }


    # ========================================================
    # STEP 11 — CACHE VERIFIED RESULT
    # ========================================================

    CACHE[cache_key] = result


    # ========================================================
    # STEP 12 — AUDIT COMPLETION
    # ========================================================

    log_event(
        incident.incident_id,
        "investigation_completed",
        "success"
    )

    return result